In [ ]:
#! pip install duckdb

In [3]:
import requests
import duckdb
from pathlib import Path

In [ ]:

db_path = "http://leg.ufpr.br/~wagner/data/music_decoder_nacional.duckdb"
destination_path = "music_decoder_nacional.duckdb"

if not Path(destination_path).exists():
    # Download to local
    response = requests.get(db_path)
    with open(destination_path, "wb") as f:
        f.write(response.content)

con = duckdb.connect("music_decoder_nacional.duckdb")


In [ ]:


query = "SELECT * FROM info_musicas"
df = con.execute(query).fetchdf()
display(df.head())


,songstats_track_id,release_date,title,artistas,N_artistas,colaboradores,papel_colaborador,N_colaboradores,labels,N_labels,distribuidor,N_distribuidor,genero
0,qaefgb2n,2020-07-01,- Para Ser,Luiz Caldas,1,Gerônimo,Composer | Songwriter,1,,0,ONErpm,1,
1,x21esgu9,2022-07-01,"¡Basta, No Llores Más!",Luiz Caldas,1,Jorge Zárath,Composer | Songwriter,1,,0,ONErpm,1,latin | worldwide | pop latino | south america
2,5kltg9p2,2001-01-01,(Desperta) Preconceito de Cor,Margareth Menezes,1,Margareth Menezes da Purificação,Composer,1,,0,WMG,1,brazilian
3,y9az03bq,2015-05-12,(Everything I Do) I Do It For You - Ao Vivo,Zezé Di Camargo & Luciano,1,Mutt Lange | Michael Karmen | Brayn Adams,Composer | Songwriter | Composer | Songwriter ...,3,,0,Sony Music,1,sertanejo | brazilian
4,ctk2g8lr,2019-11-29,@Isa - Ao Vivo,Marcos & Belutti,1,Isabella Resende | Danillo Davílla | Cristyan ...,Composer | Composer | Composer | Producer | Co...,5,,0,Sony Music,1,sertanejo | brazilian | sertanejo universitrio


In [8]:


query = "SELECT * FROM historico_musicas_spotify"
df = con.execute(query).fetchdf()
display(df.head())


,data_mensal,date_min,streams_total_min,popularity_current_min,mes_min,ano_min,source,songstats_id,nome_musica,nome_artista,artista_songstats_id
0,2022-01-01,19010.0,392289.0,0.0,1.0,2022.0,spotify,y9az03bq,(Everything I Do) I Do It For You - Ao Vivo,Zezé Di Camargo & Luciano,g6b41h3i
1,2022-02-01,19024.0,392289.0,0.0,2.0,2022.0,spotify,y9az03bq,(Everything I Do) I Do It For You - Ao Vivo,Zezé Di Camargo & Luciano,g6b41h3i
2,2022-03-01,19052.0,393196.0,0.0,3.0,2022.0,spotify,y9az03bq,(Everything I Do) I Do It For You - Ao Vivo,Zezé Di Camargo & Luciano,g6b41h3i
3,2022-04-01,19083.0,394883.0,0.0,4.0,2022.0,spotify,y9az03bq,(Everything I Do) I Do It For You - Ao Vivo,Zezé Di Camargo & Luciano,g6b41h3i
4,2022-05-01,19113.0,394883.0,0.0,5.0,2022.0,spotify,y9az03bq,(Everything I Do) I Do It For You - Ao Vivo,Zezé Di Camargo & Luciano,g6b41h3i


In [9]:


query = "SELECT DISTINCT genero FROM info_musicas"
df = con.execute(query).fetchdf()
display(df.head())


,genero
0,latin | worldwide
1,country | rock | sertanejo | brazilian
2,sertanejo | forró | brazilian
3,pop | agronejo | swing music | academic | chee...
4,pop | mpb | brazilian


In [108]:

query = """
    SELECT DISTINCT
        songstats_id,
        nome_musica,
        nome_artista
    FROM historico_musicas_spotify hms
"""

df = con.execute(query).fetchdf()
df.head()

,songstats_id,nome_musica,nome_artista
0,sonv92p5,Amor Fatal - Acústico,Calcinha Preta
1,oxz6s2au,Amor Maior,Tarcísio do Acordeon
2,b84fxyrz,Amor ou Esquema,Toque Dez
3,nxh5auyi,Amor ou Paixão,Walkyria Santos
4,vycmrifo,Amor Primeiro,Calcinha Preta


In [112]:
import pandas as pd

files = [
    (Path(".") / "abt" / "3_abt_song_lyric_views_sertanejo.parquet"),
    (Path(".") / "abt" / "3_abt_song_lyric_views_forro.parquet")
]

df_letras = pd.read_parquet(files)
df_letras.head()


,genre,index,title,artist,link,n_views,lyrics,paragraph
0,sertanejo,1,Evidências,Chitãozinho & Xororó,/chitaozinho-e-xororo/768469/,8268004,Quando eu digo que deixei de te amar\nÉ porque...,1
1,sertanejo,1,Evidências,Chitãozinho & Xororó,/chitaozinho-e-xororo/768469/,8268004,Eu me afasto e me defendo de você\nMas depois ...,2
2,sertanejo,1,Evidências,Chitãozinho & Xororó,/chitaozinho-e-xororo/768469/,8268004,E nessa loucura de dizer que não te quero\nVou...,3
3,sertanejo,1,Evidências,Chitãozinho & Xororó,/chitaozinho-e-xororo/768469/,8268004,Chega de mentiras\nDe negar o meu desejo\nEu t...,4
4,sertanejo,1,Evidências,Chitãozinho & Xororó,/chitaozinho-e-xororo/768469/,8268004,"Diz que é verdade, que tem saudade\nQue ainda ...",5


In [113]:

# Aggregate lyrics
df_letras["letra_agg"] = (
    df_letras
    .sort_values("paragraph", ascending=True)
    .groupby(["genre", "index"])
    ["lyrics"]
    .transform(
        lambda x: "\n\n".join(x)
    )
)

df_letras.drop(columns=["paragraph", "lyrics"], inplace=True)
df_letras.drop_duplicates(inplace=True)
df_letras.rename(columns={"letra_agg": "lyrics"}, inplace=True)
df_letras.reset_index(drop=True, inplace=True)
df_letras.head()


,genre,index,title,artist,link,n_views,lyrics
0,sertanejo,1,Evidências,Chitãozinho & Xororó,/chitaozinho-e-xororo/768469/,8268004,Quando eu digo que deixei de te amar\nÉ porque...
1,sertanejo,2,Te Esperando,Luan Santana,/luan-santana/te-esperando/,7140618,Mesmo que você não caia na minha cantada\nMesm...
2,sertanejo,3,Foi por Conveniência,Marília Mendonça,/marilia-mendonca/conveniencia/,197280,"Bonito não é, nem chega aos pés\nDo conto de f..."
3,sertanejo,4,Caso Indefinido,Cristiano Araújo,/cristiano-araujo/caso-indefinido/,5105678,Será que alguém explica a nossa relação?\nUm c...
4,sertanejo,5,Decida,Milionário e José Rico,/milionario-e-jose-rico/83421/,1747608,"No amor, há momentos\nQue temos que dizer um p..."


In [130]:

import unicodedata

def remove_accents(text):    
    normalized = unicodedata.normalize('NFD', text)
    text = ''.join(c for c in normalized if unicodedata.category(c) != 'Mn')
    return text

def formatar_chave(title):
    if not title:
        return None
    if " - " in title:
        title = title.split(" - ")
    if " / " in title:
        title = title.split(" / ")
    if " \\ " in title:
        title = title.split(" \\ ")
    if " | " in title:
        title = title.split(" | ")
    if isinstance(title, list):
        title = title[0]
        title = remove_accents(title)
    if isinstance(title, str):
        title = remove_accents(title)
        return title.replace(" ", "_").lower()

    return None

df_letras["_key_title"] = df_letras["title"].apply(formatar_chave)
df_letras["_key_artist"] = df_letras["artist"].apply(formatar_chave)

df["_key_title"] = df["nome_musica"].apply(formatar_chave)
df["_key_artist"] = df["nome_artista"].apply(formatar_chave)


In [138]:

df_unida = (
    df_letras.set_index(["_key_title", "_key_artist"])
    .join(
        df.set_index(["_key_title", "_key_artist"]),
        how="inner",
        rsuffix="_spotify"
    )
)

df_unida.reset_index(drop=False, inplace=True)

print(f"Total de músicas na tabela unida: {len(df_unida)}")
df_unida.head(2)

Total de músicas na tabela unida: 986


,_key_title,_key_artist,genre,index,title,artist,link,n_views,lyrics,_key,songstats_id,nome_musica,nome_artista,_key_spotify
0,evidencias,chitaozinho_&_xororo,sertanejo,1,Evidências,Chitãozinho & Xororó,/chitaozinho-e-xororo/768469/,8268004,Quando eu digo que deixei de te amar\nÉ porque...,evidências,r6dh4lju,Evidências,Chitãozinho & Xororó,evidências
1,evidencias,chitaozinho_&_xororo,sertanejo,1,Evidências,Chitãozinho & Xororó,/chitaozinho-e-xororo/768469/,8268004,Quando eu digo que deixei de te amar\nÉ porque...,evidências,gdkcy61e,Evidências,Chitãozinho & Xororó,evidências


In [134]:

df_unida[["title", "artist"]].value_counts()


title                 artist              
Pelas Ruas Que Andei  Alceu Valença           6
Coração Bobo          Alceu Valença           5
Evidências            Chitãozinho & Xororó    5
Mar de Doçura         Cavaleiros do Forró     5
Inevitável            Bruno & Marrone         5
                                             ..
A Culpa É Nossa       Maiara & Maraisa        1
A Casa Caiu           Calcinha Preta          1
A Carta               Eduardo Costa           1
A                     Luan Santana            1
60 Segundos           Gusttavo Lima           1
Name: count, Length: 697, dtype: int64

In [198]:

df_final = con.query("""
          
    WITH 
        base_sem_duplicacoes AS (
            SELECT DISTINCT
                *
            FROM historico_musicas_spotify
        ),
        data_corte AS (
            SELECT 
                songstats_id,
                DATE_ADD(MIN(data_mensal), INTERVAL 6 MONTH) AS data_corte
            FROM historico_musicas_spotify
            WHERE streams_total_min > 0 
          GROUP BY 
            songstats_id
        )
          
    SELECT 
        hms.streams_total_min,
        dfu.*
    FROM df_unida dfu
    LEFT JOIN base_sem_duplicacoes hms ON (
        dfu.songstats_id = hms.songstats_id
    )
    INNER JOIN data_corte dc ON (
        hms.songstats_id = dc.songstats_id
        AND hms.data_mensal = dc.data_corte
    )
        
""").fetchdf()

df_final.head()


,streams_total_min,_key_title,_key_artist,genre,index,title,artist,link,n_views,lyrics,_key,songstats_id,nome_musica,nome_artista,_key_spotify
0,5737715.0,agora_e_pra_valer,wesley_safadao,forro,286,Agora É Pra Valer,Wesley Safadão,/wesley-safadao/agora-e-pra-valer/,152543,Eu vou me dar um tempo\nPra ajeitar as minhas ...,agora_é_pra_valer,5ol4ds2j,Agora é Pra Valer - Ao Vivo,Wesley Safadão,Agora é Pra Valer
1,707525.0,amor_da_minha_vida,calcinha_preta,forro,308,Amor da Minha Vida,Calcinha Preta,/calcinha-preta/102461/,111972,Amor da minha vida\nNão da pra te esquecer eu ...,amor_da_minha_vida,jx4qze58,Amor da Minha Vida - Ao Vivo,Calcinha Preta,Amor da Minha Vida
2,13061122.0,flashback,gustavo_mioto,sertanejo,839,Flashback,Gustavo Mioto,/gustavo-mioto/flashback/,163052,"Eu tento não lembrar, mas lembro\nNão tente me...",flashback,70tpyu41,Flashback - Ao Vivo,Gustavo Mioto,Flashback
3,212017.0,inevitavel,bruno_&_marrone,sertanejo,204,Inevitável,Bruno & Marrone,/bruno-e-marrone/81266/,569226,É animal\nÉ tão voraz essa paixão\nÉ vendaval\...,inevitável,qjlvotfb,Inevitável - Ao Vivo,Bruno & Marrone,Inevitável
4,148943.0,faco_chover,calcinha_preta,forro,36,Faço Chover,Calcinha Preta,/calcinha-preta/2002382/,709329,"Se você me quer ainda\nMe avise, por favor\nVo...",faço_chover,nzf0sbd3,Faço Chover,Calcinha Preta,faço_chover


In [201]:

df_final.to_parquet(Path(".") / "abt" / "3_1_abt_song_lyric_spotify_listeners_6_months.parquet", index=False)
